# Task 1: Daily Return Analysis

### Objective
Compute daily returns for all funds: daily_return = nav_t / nav_t-1 - 1, Annualised return = (1 +
daily_return).prod()^(252/n) - 1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
nav = pd.read_csv("../data/raw/02_nav_history.csv")
benchmark = pd.read_csv("../data/raw/10_benchmark_indices.csv")
performance = pd.read_csv("../data/raw/07_scheme_performance.csv")

In [ ]:
returns_data = nav.copy()
returns_data["date"] = pd.to_datetime(returns_data["date"])
returns_data = returns_data.sort_values(["amfi_code", "date"])
returns_data["daily_return"] = (returns_data.groupby("amfi_code")["nav"].pct_change())
returns_data = returns_data.dropna(subset=["daily_return"])
returns_data.head()

In [ ]:
print("Number of Funds:",returns_data["amfi_code"].nunique())
print("Total Return Records:",len(returns_data))
print("Missing Values:",returns_data["daily_return"].isnull().sum())
returns_data["daily_return"].describe()

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(returns_data["daily_return"],bins=60,kde=True)
plt.title("Distribution of Daily Returns")
plt.xlabel("Daily Return")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
extreme_moves = returns_data[abs(returns_data["daily_return"]) > 0.10]
print("Observations Beyond ±10%:",len(extreme_moves))
extreme_moves.head()

In [ ]:
annualised_returns=(returns_data.groupby("amfi_code")["daily_return"]
    .apply(
        lambda x:
        ((1 + x).prod()) **
        (252 / len(x))
        - 1
    ).reset_index()
)
annualised_returns.columns = ["amfi_code","annualised_return"]
annualised_returns.head()

In [ ]:
returns_computed = returns_data.merge(annualised_returns,on="amfi_code",how="left")
returns_computed.to_csv("../data/processed/returns_computed.csv",index=False)
print("returns_computed.csv created successfully.")

# Task 2: CAGR Analysis
### Objective 
Calculate CAGR for 1yr, 3yr, 5yr periods: CAGR = (NAV_end /NAV_start) ^ (1/n) - 1 for SBI
Bluechip, HDFC Top 100, etc.


In [ ]:
cagr_data = nav.copy()
cagr_data["date"] = pd.to_datetime(cagr_data["date"])
cagr_data = cagr_data.sort_values(["amfi_code", "date"])
print("Funds:",cagr_data["amfi_code"].nunique())
cagr_data.head()

In [ ]:
def calculate_cagr(nav_start, nav_end, years):
    if nav_start <= 0 or nav_end <= 0:
        return np.nan
    return ((nav_end / nav_start) **(1 / years)) - 1

In [ ]:
cagr_results = []
for fund_code in cagr_data["amfi_code"].unique():
    fund_nav = (cagr_data[cagr_data["amfi_code"] == fund_code].sort_values("date"))
    latest_nav = fund_nav.iloc[-1]["nav"]
    result = {"amfi_code": fund_code}
    for years in [1, 3, 5]:
        trading_days = years * 252
        if len(fund_nav) > trading_days:
            historical_nav = (fund_nav.iloc[-(trading_days + 1)]["nav"])
            result[f"cagr_{years}yr"] = calculate_cagr(historical_nav,latest_nav,years)
        else:
            result[f"cagr_{years}yr"] = np.nan
    cagr_results.append(result)
cagr_table = pd.DataFrame(cagr_results)
cagr_table.head()

In [ ]:
cagr_display = cagr_table.copy()
for col in ["cagr_1yr","cagr_3yr","cagr_5yr"]:
    cagr_display[col] = (cagr_display[col] * 100).round(2)
cagr_display.head()

In [ ]:
top_cagr=(cagr_display.sort_values("cagr_3yr",ascending=False).head(10))
top_cagr

In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(data=top_cagr,x="amfi_code",y="cagr_3yr")
plt.title("Top 10 Funds by 3-Year CAGR")
plt.xlabel("AMFI Code")
plt.ylabel("3-Year CAGR (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
cagr_table.to_csv("../data/processed/cagr_comparison.csv",index=False)
print("cagr_comparison.csv saved successfully.")

# Task 3: Sharpe Ratio Analysis
### Objective
Compute Sharpe Ratio: Sharpe = (Rp- Rf) / Std(Rp) Use Rf = 6.5% (RBI repo rate proxy) Annualise with
sqrt(252)


In [ ]:
risk_free_rate = 0.065
daily_risk_free_rate = (risk_free_rate / 252)
print("Daily Risk-Free Rate:",round(daily_risk_free_rate, 8))

In [ ]:
sharpe_results = []
for fund_code in returns_data["amfi_code"].unique():
    fund_returns = (returns_data[returns_data["amfi_code"] == fund_code]["daily_return"])
    average_return = fund_returns.mean()
    return_volatility = fund_returns.std()
    if return_volatility > 0:
        sharpe_ratio = ((average_return - daily_risk_free_rate)/ return_volatility) * np.sqrt(252)
    else:
        sharpe_ratio = np.nan
    sharpe_results.append({"amfi_code": fund_code,"sharpe_ratio": sharpe_ratio})
sharpe_table = pd.DataFrame(sharpe_results)
sharpe_table.head()

In [ ]:
sharpe_table = (sharpe_table.sort_values("sharpe_ratio",ascending=False).reset_index(drop=True))
sharpe_table["rank"] = (sharpe_table.index + 1)
sharpe_table.head(10)

In [ ]:
sharpe_table["sharpe_ratio"].describe()

In [ ]:
plt.figure(figsize=(12,6))
top_sharpe = sharpe_table.head(10)
sns.barplot(data=top_sharpe,x="amfi_code",y="sharpe_ratio")
plt.title("Top 10 Funds by Sharpe Ratio")
plt.xlabel("AMFI Code")
plt.ylabel("Sharpe Ratio")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
sharpe_table.to_csv("../data/processed/sharpe_ratio_results.csv",index=False)
print("sharpe_ratio_results.csv saved successfully.")

# Task 4: Sortino Ratio Analysis
### Objective
Compute Sortino Ratio: Sortino = (Rp - Rf) / Downside_Std where Downside_Std uses only negative
return days


In [ ]:
risk_free_rate = 0.065
daily_risk_free_rate = (risk_free_rate / 252)
print("Daily Risk-Free Rate:",round(daily_risk_free_rate, 8))

In [ ]:
sortino_results = []
for fund_code in returns_data["amfi_code"].unique():
    fund_returns = (returns_data[returns_data["amfi_code"] == fund_code]["daily_return"])
    average_daily_return = (fund_returns.mean())
    downside_returns = (fund_returns[fund_returns < 0])
    downside_deviation = (downside_returns.std())
    if (pd.notna(downside_deviation)and downside_deviation > 0):
        sortino_ratio = ((average_daily_return- daily_risk_free_rate)/ downside_deviation) * np.sqrt(252)
    else:
        sortino_ratio = np.nan
    sortino_results.append({"amfi_code": fund_code,"sortino_ratio": sortino_ratio})
sortino_table = pd.DataFrame(sortino_results)
sortino_table.head()

In [ ]:
sortino_table = (sortino_table.sort_values("sortino_ratio",ascending=False).reset_index(drop=True))
sortino_table["sortino_rank"]=(sortino_table.index + 1)
sortino_table.head(10)

In [ ]:
print("Funds Evaluated:",len(sortino_table))
sortino_table["sortino_ratio"].describe()

In [ ]:
top_sortino = (sortino_table.head(10))
plt.figure(figsize=(12,6))
sns.barplot(data=top_sortino,x="amfi_code",y="sortino_ratio")
plt.title("Top 10 Funds by Sortino Ratio")
plt.xlabel("AMFI Code")
plt.ylabel("Sortino Ratio")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
sortino_table.to_csv("../data/processed/sortino_ratio_results.csv",index=False)
print("sortino_ratio_results.csv saved successfully.")

# Task 5: Alpha and Beta Analysis
### Objective
Compute Alpha & Beta vs benchmark: Regress fund returns on Nifty 100 returns (OLS) Alpha = intercept * 252, Beta = slope Use scipy.stats.linregress

In [ ]:
benchmark_data = benchmark.copy()
benchmark_data["date"] = pd.to_datetime(benchmark_data["date"])
benchmark_data = benchmark_data[benchmark_data["index_name"] == "NIFTY100"]
benchmark_data = benchmark_data.sort_values("date")
benchmark_data["benchmark_return"] = (benchmark_data["close_value"].pct_change())
benchmark_data = benchmark_data.dropna()
benchmark_data.head()

In [ ]:
alpha_beta_results = []
for fund_code in returns_data["amfi_code"].unique():
    fund_returns = (returns_data[returns_data["amfi_code"] == fund_code][["date", "daily_return"]])
    merged_returns = pd.merge(fund_returns,benchmark_data[["date", "benchmark_return"]],on="date",how="inner")
    if len(merged_returns) > 30:
        regression_model = linregress(merged_returns["benchmark_return"],
            merged_returns["daily_return"]
        )

        alpha_beta_results.append({
            "amfi_code": fund_code,
            "alpha": regression_model.intercept * 252,
            "beta": regression_model.slope
        })
alpha_beta_table = pd.DataFrame(alpha_beta_results)
alpha_beta_table.head()

In [ ]:
print("Funds Processed:",len(alpha_beta_table))
alpha_beta_table.describe()

In [ ]:
benchmark_data.head()

In [ ]:
alpha_beta_table.head()

In [ ]:
alpha_beta_table.to_csv("../data/processed/alpha_beta.csv",index=False)
print("alpha_beta.csv saved successfully.")

# Task 6: Maximum Drawdown Analysis
### Objective
Compute Maximum Drawdown: max_dd = min(NAV / running_max - 1) Highlight worst drawdown period
for each fund

In [ ]:
drawdown_results = []
for fund_code in nav["amfi_code"].unique():
    fund_data = (nav[nav["amfi_code"] == fund_code]
        .copy()
        .sort_values("date")
    )
    fund_data["running_peak"] = (fund_data["nav"].cummax())
    fund_data["drawdown"] = (fund_data["nav"]/ fund_data["running_peak"]) - 1
    worst_record = fund_data.loc[fund_data["drawdown"].idxmin()]
    drawdown_results.append({
        "amfi_code": fund_code,
        "max_drawdown": worst_record["drawdown"],
        "drawdown_date": worst_record["date"]
    })
drawdown_table = pd.DataFrame(drawdown_results)
drawdown_table.head()

In [ ]:
drawdown_table = (drawdown_table.sort_values("max_drawdown",ascending=False)
    .reset_index(drop=True))
drawdown_table["drawdown_rank"] = (drawdown_table.index + 1)
drawdown_table.head(10)

In [ ]:
print("Funds Evaluated:",len(drawdown_table))
print("\nDrawdown Summary")
drawdown_table["max_drawdown"].describe()

In [ ]:
best_drawdown = (drawdown_table.head(10))
plt.figure(figsize=(12,6))
sns.barplot(data=best_drawdown,x="amfi_code",y="max_drawdown")
plt.title("Top 10 Funds with Lowest Maximum Drawdown")
plt.xlabel("AMFI Code")
plt.ylabel("Maximum Drawdown")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
worst_drawdowns = (drawdown_table.sort_values("max_drawdown").head(10))
worst_drawdowns

In [ ]:
drawdown_table.to_csv("../data/processed/max_drawdown_results.csv",index=False)
print("max_drawdown_results.csv saved successfully.")

# Task 7: Fund Scorecard
### Objective
Build Fund Scorecard (composite score 0-100): Score = 30%×(3yr return rank) + 25%×(Sharpe rank) +
20%×(Alpha rank) + 15%×(Expense ratio rank, inverse) + 10%×(Max DD rank, inverse)

In [ ]:
fund_scorecard_data = (cagr_table[["amfi_code", "cagr_3yr"]]
    .merge(sharpe_table[["amfi_code", "sharpe_ratio"]],on="amfi_code")
    .merge(alpha_beta_table[["amfi_code", "alpha"]],on="amfi_code")
    .merge(drawdown_table[["amfi_code", "max_drawdown"]],on="amfi_code")
)
fund_scorecard_data.head()

In [ ]:
expense_ratio_data = (performance[["amfi_code", "expense_ratio_pct"]])
fund_scorecard_data = (fund_scorecard_data
    .merge(
        expense_ratio_data,
        on="amfi_code",
        how="left"
    )
)
fund_scorecard_data.head()

In [ ]:
fund_scorecard_data["return_rank"] = (fund_scorecard_data["cagr_3yr"]
    .rank(ascending=False,method="dense")
)
fund_scorecard_data["sharpe_rank"] = (fund_scorecard_data["sharpe_ratio"]
    .rank(ascending=False,method="dense")
)
fund_scorecard_data["alpha_rank"] = (fund_scorecard_data["alpha"]
    .rank(ascending=False,method="dense")
)

fund_scorecard_data["expense_rank"] = (fund_scorecard_data["expense_ratio_pct"]
    .rank(ascending=True,method="dense")
)

fund_scorecard_data["drawdown_rank"] = (fund_scorecard_data["max_drawdown"]
    .rank(ascending=False,method="dense")
)

In [ ]:
max_rank = (
    fund_scorecard_data[["return_rank","sharpe_rank","alpha_rank","expense_rank","drawdown_rank"]]
    .max().max()
)
fund_scorecard_data["return_score"] = ((max_rank -fund_scorecard_data["return_rank"] + 1)/ max_rank)
fund_scorecard_data["sharpe_score"] = ((max_rank -fund_scorecard_data["sharpe_rank"] + 1)/ max_rank)
fund_scorecard_data["alpha_score"] = ((max_rank -fund_scorecard_data["alpha_rank"] + 1)/ max_rank)
fund_scorecard_data["expense_score"] = ((max_rank -fund_scorecard_data["expense_rank"] + 1)/ max_rank)
fund_scorecard_data["drawdown_score"] = ((max_rank -fund_scorecard_data["drawdown_rank"] + 1)/ max_rank)

In [ ]:
fund_scorecard_data["fund_score"] = (
      fund_scorecard_data["return_score"] * 30
    + fund_scorecard_data["sharpe_score"] * 25
    + fund_scorecard_data["alpha_score"] * 20
    + fund_scorecard_data["expense_score"] * 15
    + fund_scorecard_data["drawdown_score"] * 10
)
fund_scorecard_data["fund_score"] = (fund_scorecard_data["fund_score"].round(2))

In [ ]:
fund_scorecard = (fund_scorecard_data
    .sort_values(
        "fund_score",
        ascending=False
    ).reset_index(drop=True)
)

fund_scorecard["overall_rank"] = (fund_scorecard.index + 1)
fund_scorecard.head(10)

In [ ]:
top_funds = (fund_scorecard.head(10))
plt.figure(figsize=(12,6))
sns.barplot(
    data=top_funds,
    x="amfi_code",
    y="fund_score"
)
plt.title("Top 10 Funds by Composite Score")
plt.xlabel("AMFI Code")
plt.ylabel("Fund Score")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
fund_scorecard.to_csv("../data/processed/fund_scorecard.csv",index=False)
print("fund_scorecard.csv saved successfully.")

# Task 8: Benchmark Comparison
### Objective
Benchmark comparison chart: Plot top 5 funds vs Nifty 50 and Nifty 100 over 3 years. Compute tracking error.

In [ ]:
top_funds = (fund_scorecard.head(5)["amfi_code"].tolist())
print("Top 5 Funds:")
print(top_funds)
nav["date"] = pd.to_datetime(nav["date"])

In [ ]:
benchmark_chart = benchmark.copy()
benchmark_chart["date"] = pd.to_datetime(benchmark_chart["date"])
benchmark_chart = benchmark_chart[benchmark_chart["index_name"].isin(["NIFTY50", "NIFTY100"])]
benchmark_chart = benchmark_chart.sort_values(["index_name", "date"])
benchmark_chart.head()

In [ ]:
latest_date = nav["date"].max()
start_date = (latest_date - pd.DateOffset(years=3))
fund_nav_3yr = nav[(nav["date"] >= start_date)&(nav["amfi_code"].isin(top_funds))].copy()
benchmark_3yr = benchmark_chart[benchmark_chart["date"] >= start_date].copy()
print("Analysis Start Date:")
print(start_date.date())

In [ ]:
fund_nav_3yr["normalized_value"] = (fund_nav_3yr.groupby("amfi_code")["nav"].transform(
        lambda x: (
            x / x.iloc[0]
        ) * 100
    )
)

In [ ]:
benchmark_3yr["normalized_value"] = (benchmark_3yr.groupby("index_name")["close_value"]
    .transform(
        lambda x: (
            x / x.iloc[0]
        ) * 100
    )
)

In [ ]:
plt.figure(figsize=(14,7))
for fund in top_funds:
    fund_subset = fund_nav_3yr[fund_nav_3yr["amfi_code"] == fund]
    plt.plot(
        fund_subset["date"],
        fund_subset["normalized_value"],
        label=f"Fund {fund}"
    )
for benchmark_name in ["NIFTY50","NIFTY100"]:
    benchmark_subset = benchmark_3yr[benchmark_3yr["index_name"] == benchmark_name]
    plt.plot(
        benchmark_subset["date"],
        benchmark_subset["normalized_value"],
        linewidth=3,
        label=benchmark_name
    )
plt.title("Top Funds vs Market Benchmarks (3 Years)")
plt.xlabel("Date")
plt.ylabel("Indexed Growth (Base = 100)")
plt.legend()
plt.tight_layout()
plt.savefig("../reports/benchmark_comparison.png",bbox_inches="tight")
plt.show()

In [ ]:
tracking_error_results = []
nifty100_returns = benchmark_3yr[
    benchmark_3yr["index_name"] == "NIFTY100"
][["date", "close_value"]].copy()

nifty100_returns["benchmark_return"] = (
    nifty100_returns["close_value"]
    .pct_change()
)
nifty100_returns = nifty100_returns.dropna()

In [ ]:
for fund_code in top_funds:
    fund_returns = (
        returns_data[
            returns_data["amfi_code"] == fund_code
        ][["date", "daily_return"]]
    )
    merged_returns = pd.merge(
        fund_returns,
        nifty100_returns[
            ["date", "benchmark_return"]
        ],
        on="date",
        how="inner"
    )
    tracking_error = ((merged_returns["daily_return"]-merged_returns["benchmark_return"])
        .std()*np.sqrt(252))
    tracking_error_results.append({"amfi_code": fund_code,"tracking_error": tracking_error})
tracking_error_table = pd.DataFrame(tracking_error_results)
tracking_error_table

In [ ]:
tracking_error_table.to_csv("../data/processed/tracking_error_results.csv",index=False)
print("benchmark_comparison.png saved.")
print("tracking_error_results.csv saved.")